In [1]:
import os 
import requests , json 
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
# langchain import 
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader ,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
# vector store
from langchain_community.vectorstores import Chroma

C:\Users\vinee\AppData\Local\Temp\ipykernel_22196\1585745039.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader ,TextLoader
d:\Project\KrishnaRAGUdemy\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sample_docs = [
    {
        "title": "What is Python?",
        "description": (
            "Python ek high-level, interpreted programming language hai jo readability aur developer "
            "productivity ke liye design ki gayi hai. Iska syntax itna clean hota hai ki beginners bhi "
            "jaldi seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem "
            "bahut bada hai — data engineering, machine learning, automation, APIs, cloud, aur DevOps sab "
            "me use hota hai. Iski libraries jaise pandas, pyspark, numpy, fastapi, airflow, aur databricks "
            "runtime Python ko enterprise-grade workflows ke liye perfect banati hain."
        )
    },
    {
        "title": "Why Python is Popular?",
        "description": (
            "Python ki popularity ka sabse bada reason iska simplicity-first design philosophy hai. "
            "Developers ko boilerplate code likhne ki zaroorat nahi padti, aur complex logic ko bhi "
            "few lines me express kiya ja sakta hai. Python cross-platform hai, open-source hai, aur "
            "community support unmatched hai. Data engineers ke liye Python ek backbone language ban chuki "
            "hai kyunki ye ETL pipelines, streaming jobs, orchestration, cloud automation, aur distributed "
            "processing frameworks ke saath seamlessly integrate hoti hai."
        )
    },
    {
        "title": "Where Python is Used?",
        "description": (
            "Python ka use-case spectrum bahut wide hai. Data engineering me Python ka use ingestion pipelines, "
            "Kafka streaming, Spark transformations, Delta Lake processing, aur orchestration tools jaise "
            "Airflow me hota hai. Machine learning me Python TensorFlow, PyTorch, aur Scikit-learn jaise "
            "frameworks ko power deta hai. Web development me FastAPI aur Django popular frameworks hain. "
            "Automation, scripting, cloud infrastructure provisioning, aur DevOps workflows me bhi Python "
            "industry standard ban chuka hai."
        )
    }
]


In [4]:
base_path= os.getcwd()
folder_name="docs"
folder_path = os.path.join(base_path,folder_name)
if not os.path.exists(folder_path):
    os.mkdir(folder_path)
    print('Path is created successfully')
else:
    print('Folder already exists')

Folder already exists


In [5]:
for i ,doc in enumerate(sample_docs):
    with open(f'{folder_path}/doc_{i+1}.txt', 'w',encoding='utf-8') as f:
        content = doc.get('description','')
        print(content)
        f.write(content)

Python ek high-level, interpreted programming language hai jo readability aur developer productivity ke liye design ki gayi hai. Iska syntax itna clean hota hai ki beginners bhi jaldi seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem bahut bada hai — data engineering, machine learning, automation, APIs, cloud, aur DevOps sab me use hota hai. Iski libraries jaise pandas, pyspark, numpy, fastapi, airflow, aur databricks runtime Python ko enterprise-grade workflows ke liye perfect banati hain.
Python ki popularity ka sabse bada reason iska simplicity-first design philosophy hai. Developers ko boilerplate code likhne ki zaroorat nahi padti, aur complex logic ko bhi few lines me express kiya ja sakta hai. Python cross-platform hai, open-source hai, aur community support unmatched hai. Data engineers ke liye Python ek backbone language ban chuki hai kyunki ye ETL pipelines, streaming jobs, orchestration, cloud automation, aur distributed processing framew

In [6]:
diretoryLoader= DirectoryLoader(path=folder_path,
                                loader_cls=TextLoader,
                                 loader_kwargs={'encoding': 'utf-8'},
                                glob="*.txt")


In [7]:
documents = diretoryLoader.load()

In [8]:
rescurseSplitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n", " ", ""], chunk_size=200 , chunk_overlap=20 , length_function=len )  
chunks = rescurseSplitter.split_documents(documents)

print("Total chunks:", len(chunks))
for i, c in enumerate(chunks):
    print(f"Chunk {i+1} length:", c.metadata ,  c.page_content)


Total chunks: 9
Chunk 1 length: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'} Python ek high-level, interpreted programming language hai jo readability aur developer productivity ke liye design ki gayi hai. Iska syntax itna clean hota hai ki beginners bhi jaldi seekh lete hain,
Chunk 2 length: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'} seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem bahut bada hai — data engineering, machine learning, automation, APIs, cloud, aur DevOps sab me use hota hai.
Chunk 3 length: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'} me use hota hai. Iski libraries jaise pandas, pyspark, numpy, fastapi, airflow, aur databricks runtime Python ko enterprise-grade workflows ke liye perfect banati hain.
Chunk 4 length: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_2.txt'} Python ki popularity ka sabse

In [9]:
print(len(documents))
for doc in documents:
    print(f"{doc.metadata}", {doc.page_content[0:20]})
    


3
{'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'} {'Python ek high-level'}
{'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_2.txt'} {'Python ki popularity'}
{'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_3.txt'} {'Python ka use-case s'}


### Embeddings
#### HuggingFace

In [10]:
## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" , show_progress=True
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2811.40it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=True)

In [11]:
sample_text= "Hi this is a lord shiva  creator or the earth "
vector = embeddings.embed_query(sample_text)
print(f'Vector created {len(vector)} and vector is {vector[:5]}')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.86it/s]

Vector created 384 and vector is [-0.02587340772151947, 0.05436141416430473, 0.01602882519364357, 0.044871143996715546, -0.040070269256830215]


#### Documents Vectors

In [12]:
chunks

[Document(metadata={'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, page_content='Python ek high-level, interpreted programming language hai jo readability aur developer productivity ke liye design ki gayi hai. Iska syntax itna clean hota hai ki beginners bhi jaldi seekh lete hain,'),
 Document(metadata={'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, page_content='seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem bahut bada hai — data engineering, machine learning, automation, APIs, cloud, aur DevOps sab me use hota hai.'),
 Document(metadata={'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, page_content='me use hota hai. Iski libraries jaise pandas, pyspark, numpy, fastapi, airflow, aur databricks runtime Python ko enterprise-grade workflows ke liye perfect banati hain.'),
 Document(metadata={'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabas

#### VectorStore Creation 

In [13]:

persist_directory = ".chroma.db"

In [14]:
vectorStore = Chroma.from_documents(documents=chunks,
                                     embedding=embeddings,
                                     persist_directory=persist_directory,
                                     collection_name="rag_collection"
                                     )


print(vectorStore._collection.count())

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.83it/s]


9


#### Similarity Search


In [15]:
query= "What is Python?"
output = vectorStore.similarity_search(query, k=3)
print("Search results:")
for i, doc in enumerate(output):
    print(f"Result {i+1}:,metadata: {doc.metadata}, content: {doc.page_content[:100]}...")

Batches: 100%|██████████| 1/1 [00:00<00:00, 56.97it/s]

Search results:
Result 1:,metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_2.txt'}, content: me express kiya ja sakta hai. Python cross-platform hai, open-source hai, aur community support unma...
Result 2:,metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, content: Python ek high-level, interpreted programming language hai jo readability aur developer productivity...
Result 3:,metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, content: seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem bahut bada ha...


### Similarity Score
#### default L2 Ecucledia lower the better

In [16]:
query= "What is Python?"
output = vectorStore.similarity_search_with_score(query, k=3)
print("Search results:")
for i, (doc, score) in enumerate(output):
    print(f"Result {i+1}: Score: {score}, Metadata: {doc.metadata}, Content: {doc.page_content[:100]}...")

Batches: 100%|██████████| 1/1 [00:00<00:00, 33.18it/s]

Search results:
Result 1: Score: 0.9812057018280029, Metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_2.txt'}, Content: me express kiya ja sakta hai. Python cross-platform hai, open-source hai, aur community support unma...
Result 2: Score: 0.995201051235199, Metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, Content: Python ek high-level, interpreted programming language hai jo readability aur developer productivity...
Result 3: Score: 1.1474753618240356, Metadata: {'source': 'd:\\Project\\KrishnaRAGUdemy\\Code_VectorDatabase\\docs\\doc_1.txt'}, Content: seekh lete hain, aur experts complex systems build kar sakte hain. Python ka ecosystem bahut bada ha...


#### LLM Initialization 
####  prompt templated design
#### fusion of all retrieved chunks to the context 

In [3]:
from langchain_huggingface import HuggingFaceEndpoint
import os 
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
from langchain_huggingface import ChatHuggingFace ,HuggingFaceEndpoint

llm_endpoint = HuggingFaceEndpoint(repo_id="meta-llama/Meta-Llama-3-8B-Instruct"
                                   ,temperature=0.7
                                   ,max_new_tokens=512
                                   ,task="text-generation")

llm_endpoint
llm = ChatHuggingFace(llm=llm_endpoint)
llm

ChatHuggingFace(output_version=None, llm=HuggingFaceEndpoint(repo_id='meta-llama/Meta-Llama-3-8B-Instruct', huggingfacehub_api_token=None, temperature=0.7, stop_sequences=[], server_kwargs={}, model_kwargs={}, model='meta-llama/Meta-Llama-3-8B-Instruct', client=<InferenceClient(model='meta-llama/Meta-Llama-3-8B-Instruct', timeout=120)>, async_client=<InferenceClient(model='meta-llama/Meta-Llama-3-8B-Instruct', timeout=120)>, task='text-generation'), model_id='meta-llama/Meta-Llama-3-8B-Instruct', temperature=0.7, top_p=0.95, max_tokens=512, model_kwargs={})

In [5]:
query="what is machine learning"
response = llm.invoke(query)

print(response.content[:200])

Machine learning is a subfield of artificial intelligence (AI) that involves training algorithms to learn from data, improve their performance, and make predictions or take actions without being expli
